In [0]:
# Test pour vérifier si Python fonctionne :
print(spark.version)

4.1.0


In [0]:
# Cellule 1 : Chargement des données Steam récupérées depuis S3 
df = spark.read.json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")

# Afficher le schéma
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
from pyspark.sql import functions as F

# On aplatit uniquement les colonnes qui EXISTENT dans le schéma
df_flat = df.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.developer").alias("developer"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.genre").alias("genre"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.owners").alias("owners"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.price").alias("price"),
    F.col("data.discount").alias("discount"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.languages").alias("languages"),
    F.col("data.categories").alias("categories"),
    F.col("data.platforms.windows").alias("windows"),
    F.col("data.platforms.mac").alias("mac"),
    F.col("data.platforms.linux").alias("linux"),
    F.col("data.ccu").alias("ccu"),
    F.col("data.type").alias("type"),
)

# Vérification
print("Nombre de jeux :", df_flat.count())
df_flat.show(3, truncate=True)

Nombre de jeux : 55691
+-------+-------+--------------+--------------------+--------------------+--------------------+------------+--------------------+--------+--------+-----+--------+------------+--------------------+--------------------+-------+-----+-----+-----+----+
|     id|  appid|          name|           developer|           publisher|               genre|release_date|              owners|positive|negative|price|discount|required_age|           languages|          categories|windows|  mac|linux|  ccu|type|
+-------+-------+--------------+--------------------+--------------------+--------------------+------------+--------------------+--------+--------+-----+--------+------------+--------------------+--------------------+-------+-----+-----+-----+----+
|     10|     10|Counter-Strike|               Valve|               Valve|              Action|   2000/11/1|10,000,000 .. 20,...|  201215|    5199|  999|       0|           0|English, French, ...|[Multi-player, Va...|   true| true

In [0]:
# Cellule 3 : Nettoyage

# 1. On garde uniquement les vrais jeux de type "game"
df_games = df_flat.filter(F.col("type") == "game")

# 2. On convertit le prix en € car il est en centimes de dollars américains
df_games = df_games.withColumn("price_eur", F.col("price")/100)

# 3. On convertit required_age en nombre entier
df_games = df_games.withColumn("required_age", F.expr("try_cast(required_age as int)"))

# 4. Créer une colonne ratio d'avis positifs (%)
# On utilise "when" pour éviter la division par zéro
df_games = df_games.withColumn(
    "positive_ratio",
    F.when(
        (F.col("positive") + F.col("negative")) > 0,
        F.round(F.col("positive") / (F.col("positive") + F.col("negative")) * 100, 2)
    ).otherwise(None)
)

# Vérification
print("Nombre de jeux après nettoyage :", df_games.count())
df_games.select("name", "price_eur", "required_age", "positive_ratio").show(5)
                                

Nombre de jeux après nettoyage : 55690
+--------------------+---------+------------+--------------+
|                name|price_eur|required_age|positive_ratio|
+--------------------+---------+------------+--------------+
|      Counter-Strike|     9.99|           0|         97.48|
|           ASCENXION|     9.99|           0|         84.38|
|         Crown Trick|     5.99|           0|         86.19|
|Cook, Serve, Deli...|    19.99|           0|          93.2|
|            细胞战争|     1.99|           0|           0.0|
+--------------------+---------+------------+--------------+
only showing top 5 rows


In [0]:
# ===================================================================================================
# ANALYSE MACRO
# Question 1 : Quel éditeur a sorti le plus de jeux sur Steam ??
# ===================================================================================================

df_publishers = df_games.groupBy("publisher") \
    .agg(F.count("appid").alias("nombre_jeux")) \
        .orderBy(F.desc("nombre_jeux"))

df_publishers.show(10, truncate=False)

+---------------+-----------+
|publisher      |nombre_jeux|
+---------------+-----------+
|Big Fish Games |422        |
|8floor         |202        |
|SEGA           |165        |
|Strategy First |151        |
|Square Enix    |141        |
|Choice of Games|140        |
|               |132        |
|Sekai Project  |132        |
|HH-Games       |132        |
|Ubisoft        |127        |
+---------------+-----------+
only showing top 10 rows


In [0]:
# Visualisation : Top des 10 éditeurs
# On retire la ligne vide
df_publishers_clean = df_publishers.filter(F.col("publisher") !="")

display(df_publishers_clean.limit(10))

publisher,nombre_jeux
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# ANALYSE DE MACRO
# Question 2 : Quels sont les jeux les mieux notés ?
# minimum 100 avis pour que cela soit représentatif)
# =================================================================================================

df_best_games = df_games.filter(F.col("positive") + F.col("negative") > 100) \
    .orderBy(F.desc("positive_ratio")) \
    .select("name", "publisher", "positive", "negative", "positive_ratio")
            
display(df_best_games.limit(10))


name,publisher,positive,negative,positive_ratio
Lucy Dreaming,Tall Story Games Ltd,118,0,100.0
Touhou Kaeizuka ～ Phantasmagoria of Flower View.,"Mediascape Co., Ltd.",119,0,100.0
Elasto Mania Remastered,Elasto Mania Team,190,0,100.0
Freshly Frosted,The Quantum Astrophysicists Guild,157,0,100.0
Distant Memoraĵo,MangaGamer,114,0,100.0
秘封旅行 ~ Secret Sealing Travel,鸽屋谷,218,0,100.0
Cyberfrags '69,D.A.S. Games,104,0,100.0
FIND ALL 2: Middle Ages,Very Very LITTLE Studio,132,0,100.0
Dragon Spirits : Prologue,indienova,109,0,100.0
Yamafuda! 2nd station,KPC,109,0,100.0


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# ANALYSE MACRO
# Question 3 : y a-t-il eu des années avec plus de sorties de jeux ?
# =================================================================================================

# On extrait l'année depuis release_date (format : 2000/11/1)
df_years = df_games.withColumn(
    "year",
    F.split(F.col("release_date"), "/").getItem(0)
).filter(
    F.col("year").rlike("^[0-9]{4}$")
).groupBy("year") \
.agg(F.count("appid").alias("nombre_sorties")) \
.orderBy("year")

display(df_years)


year,nombre_sorties
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# ANALYSE MACRO
# Question 4 : Comment les prix sont-ils distribués ?
# Y a-t-il beaucoup de jeux avec une réduction ?
# =================================================================================================

# Distribution des prix :
df_prix = df_games.filter(F.col("price_eur") > 0).select("price_eur")

display(df_prix)

price_eur
9.99
9.99
5.99
19.99
1.99
7.99
12.99
2.99
13.99
0.99


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# ANALYSE MACRO
# Question 5 : Quelles sont les langues les plus représentées ?
# =================================================================================================

df_languages = df_games.filter(F.col("languages").isNotNull()) \
    .withColumn("language", F.explode(F.split(F.col("languages"), ", "))) \
    .groupBy("language") \
    .agg(F.count("appid").alias("nombre_jeux")) \
    .orderBy(F.desc("nombre_jeux"))

display(df_languages.limit(15))

language,nombre_jeux
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

In [0]:
# ==============================================================================================
# ANALYSE MACRO
# Question 6 : Existe-t-il beaucoup de jeux
# interdits aux moins de 16/18 ans ?
# ==============================================================================================

df_age = df_games.withColumn("required_age_int", F.expr("try_cast(required_age as int)")).groupBy("required_age_int").agg(F.count("appid").alias("nombre_jeux")).orderBy("required_age_int")

display(df_age)

required_age_int,nombre_jeux
null,3
0,55029
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32


Databricks visualization. Run in Databricks to view.